# Shape Boundary Tool — Prototype Test Notebook

This notebook tests the `shape_boundary_04` MCP tool end-to-end.

**What the tool does:**  
Given either a **building shape type** (`L`, `Y`, `T`, `H`, `X`, `Z`, `O`) or a **Rhino object GUID**, it retrieves the boundary polygon of the corresponding pre-defined building footprint from Rhino.

**Test plan:**
1. Load `.env` + `mcp.json` config
2. Define the tool JSON schema
3. Validate offline shape-lookup logic (no Rhino needed)
4. Connect to MCP server (Swiftlet on `localhost:3001`)
5. Discover tools and call `shape_boundary_04` live
6. Drive the tool via an LLM prompt (Cloudflare)


## 1. Setup — Imports & Path Configuration

In [1]:
import sys, os, json, math
from pathlib import Path

# ── Ensure team_04/PY is importable ──────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

# Root of the repo (AIA26_Studio)
REPO_ROOT = NOTEBOOK_DIR.parents[1]

ENV_PATH     = REPO_ROOT / ".env"
MCP_CFG_PATH = REPO_ROOT / "mcp.json"

print(f"Repo root : {REPO_ROOT}")
print(f".env      : {ENV_PATH}  (exists={ENV_PATH.exists()})")
print(f"mcp.json  : {MCP_CFG_PATH}  (exists={MCP_CFG_PATH.exists()})")


Repo root : C:\Users\tuemi\Downloads\Glabtools\IAAC Repo\bimsc26-datamgmt-session03\AIA26_Studio
.env      : C:\Users\tuemi\Downloads\Glabtools\IAAC Repo\bimsc26-datamgmt-session03\AIA26_Studio\.env  (exists=True)
mcp.json  : C:\Users\tuemi\Downloads\Glabtools\IAAC Repo\bimsc26-datamgmt-session03\AIA26_Studio\mcp.json  (exists=True)


## 2. Load Configuration from `.env` + `mcp.json`

In [2]:
from dotenv import load_dotenv

load_dotenv(dotenv_path=ENV_PATH, override=True)

# ── Read MCP endpoint from mcp.json ──────────────────────────────────────────
with open(MCP_CFG_PATH, encoding="utf-8") as f:
    mcp_cfg = json.load(f)

servers     = mcp_cfg["mcpServers"]
server_key  = next(iter(servers))
server_cfg  = servers[server_key]

# Endpoint is stored in args[0] (SwiftletBridge pattern)
MCP_ENDPOINT = server_cfg.get("url") or server_cfg["args"][0]

# ── Read LLM credentials (Cloudflare) ────────────────────────────────────────
CF_ACCOUNT_ID = os.environ.get("CF_ACCOUNT_ID", "")
CF_API_TOKEN  = os.environ.get("CF_API_TOKEN", "")
CF_MODEL      = os.environ.get("CF_MODEL", "@cf/meta/llama-3.3-70b-instruct-fp8-fast")

CF_BASE_URL   = f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}/ai/v1"

print(f"MCP server key : {server_key}")
print(f"MCP endpoint   : {MCP_ENDPOINT}")
print(f"CF account set : {'yes' if CF_ACCOUNT_ID else 'MISSING'}")
print(f"CF token set   : {'yes' if CF_API_TOKEN else 'MISSING'}")
print(f"CF model       : {CF_MODEL}")


MCP server key : Swiftlet
MCP endpoint   : http://localhost:3001/mcp/
CF account set : yes
CF token set   : yes
CF model       : @cf/meta/llama-3.3-70b-instruct-fp8-fast


## 3. Shape Boundary Tool — JSON Schema

This is the contract the Grasshopper component must implement.  
Either `shape_type` **or** `shape_guid` is required (not both).

| Input | Type | Example | Notes |
|---|---|---|---|
| `shape_type` | string | `"L"` | Looks up pre-placed GUID from registry |
| `shape_guid` | string | `"a3b4c5d6-..."` | References any Rhino object directly |


In [3]:
SHAPE_BOUNDARY_TOOL_SCHEMA = {
    "name": "shape_boundary_04",
    "description": (
        "Retrieve the boundary polygon of a pre-defined building footprint shape from Rhino. "
        "Supply either shape_type (L/Y/T/H/X/Z/O) to look up the canonical GUID, "
        "or shape_guid to directly reference any Rhino curve object. "
        "Returns boundary coordinates, area, perimeter, centroid, and bounding box."
    ),
    "inputSchema": {
        "type": "object",
        "properties": {
            "shape_type": {
                "type": "string",
                "enum": ["L", "Y", "T", "H", "X", "Z", "O"],
                "description": "Building footprint shape code. Rhino must have a curve named/tagged with this shape pre-placed.",
            },
            "shape_guid": {
                "type": "string",
                "description": "Rhino object GUID of a curve to use directly (e.g. 'a3b4c5d6-1234-...'). Takes priority over shape_type.",
            },
        },
        # At least one of the two is required — enforced in Grasshopper logic
        "anyOf": [
            {"required": ["shape_type"]},
            {"required": ["shape_guid"]},
        ],
    },
}

print(json.dumps(SHAPE_BOUNDARY_TOOL_SCHEMA, indent=2))


{
  "name": "shape_boundary_04",
  "description": "Retrieve the boundary polygon of a pre-defined building footprint shape from Rhino. Supply either shape_type (L/Y/T/H/X/Z/O) to look up the canonical GUID, or shape_guid to directly reference any Rhino curve object. Returns boundary coordinates, area, perimeter, centroid, and bounding box.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "shape_type": {
        "type": "string",
        "enum": [
          "L",
          "Y",
          "T",
          "H",
          "X",
          "Z",
          "O"
        ],
        "description": "Building footprint shape code. Rhino must have a curve named/tagged with this shape pre-placed."
      },
      "shape_guid": {
        "type": "string",
        "description": "Rhino object GUID of a curve to use directly (e.g. 'a3b4c5d6-1234-...'). Takes priority over shape_type."
      }
    },
    "anyOf": [
      {
        "required": [
          "shape_type"
        ]
      },
    

## 4. Offline Logic Test — Shape Registry + Mock Response

Validates Python-side logic without Rhino running.  
The **shape registry** maps each shape letter to a canonical GUID.  
In Grasshopper the registry is stored as JSON on a Panel; the Python script component reads it and returns the matching boundary.


In [4]:
# ── Shape GUID registry (mirrors what lives in the Grasshopper Panel) ─────────
# Each GUID here is a placeholder that you replace with the actual Rhino object GUIDs
# after placing the footprint curves in Rhino.
SHAPE_GUID_REGISTRY: dict[str, str] = {
    "L": "00000000-0000-0000-0000-00000000000L",
    "Y": "00000000-0000-0000-0000-00000000000Y",
    "T": "00000000-0000-0000-0000-00000000000T",
    "H": "00000000-0000-0000-0000-00000000000H",
    "X": "00000000-0000-0000-0000-00000000000X",
    "Z": "00000000-0000-0000-0000-00000000000Z",
    "O": "00000000-0000-0000-0000-00000000000O",
}

# ── Mock boundary data (approximates each footprint at 1-unit grid scale) ─────
# Replace with real coordinate output from Grasshopper once curves are in Rhino.
MOCK_BOUNDARIES: dict[str, list[list[float]]] = {
    "L": [[0,0],[2,0],[2,1],[1,1],[1,3],[0,3],[0,0]],
    "Y": [[1,0],[2,1],[2,2],[1.5,2],[1.5,4],[0.5,4],[0.5,2],[0,2],[0,1],[1,0]],
    "T": [[0,0],[3,0],[3,1],[2,1],[2,3],[1,3],[1,1],[0,1],[0,0]],
    "H": [[0,0],[1,0],[1,1],[2,1],[2,0],[3,0],[3,3],[2,3],[2,2],[1,2],[1,3],[0,3],[0,0]],
    "X": [[1,0],[2,0],[2,1],[3,1],[3,2],[2,2],[2,3],[1,3],[1,2],[0,2],[0,1],[1,1],[1,0]],
    "Z": [[0,0],[3,0],[3,1],[1,1],[3,3],[0,3],[0,2],[2,2],[0,0]],
    "O": [[0,1],[1,0],[3,0],[4,1],[4,3],[3,4],[1,4],[0,3],[0,1]],
}

def _polygon_area(coords: list[list[float]]) -> float:
    """Shoelace formula — returns area of a closed polygon."""
    n = len(coords)
    area = 0.0
    for i in range(n):
        x1, y1 = coords[i]
        x2, y2 = coords[(i + 1) % n]
        area += x1 * y2 - x2 * y1
    return abs(area) / 2.0

def _polygon_perimeter(coords: list[list[float]]) -> float:
    n = len(coords)
    return sum(
        math.hypot(coords[(i+1)%n][0] - coords[i][0], coords[(i+1)%n][1] - coords[i][1])
        for i in range(n)
    )

def _centroid(coords: list[list[float]]) -> list[float]:
    xs = [p[0] for p in coords]
    ys = [p[1] for p in coords]
    return [sum(xs) / len(xs), sum(ys) / len(ys)]

def _bbox(coords: list[list[float]]) -> dict:
    xs = [p[0] for p in coords]
    ys = [p[1] for p in coords]
    return {"min": [min(xs), min(ys)], "max": [max(xs), max(ys)]}


def mock_shape_boundary(shape_type: str | None = None,
                        shape_guid: str | None = None) -> dict:
    """
    Simulates the Grasshopper shape_boundary_04 tool response.
    - shape_guid takes priority over shape_type.
    - Resolves shape_type → guid via registry, then returns mock boundary data.
    """
    if shape_guid:
        # Reverse-lookup shape type from GUID (for mock purposes)
        reverse = {v: k for k, v in SHAPE_GUID_REGISTRY.items()}
        resolved_type = reverse.get(shape_guid)
        resolved_guid = shape_guid
    elif shape_type:
        resolved_type = shape_type.upper()
        resolved_guid = SHAPE_GUID_REGISTRY.get(resolved_type)
    else:
        return {"success": False, "error": "Provide shape_type or shape_guid."}

    if resolved_type not in MOCK_BOUNDARIES:
        return {"success": False, "error": f"Unknown shape: '{resolved_type}'"}
    if not resolved_guid:
        return {"success": False, "error": f"No GUID registered for shape '{resolved_type}'"}

    coords = MOCK_BOUNDARIES[resolved_type]
    area      = _polygon_area(coords)
    perimeter = _polygon_perimeter(coords)
    centroid  = _centroid(coords)
    bbox      = _bbox(coords)

    return {
        "success": True,
        "data": {
            "shape_type":           resolved_type,
            "shape_guid":           resolved_guid,
            "boundary_coordinates": coords,
            "area_sqm":             round(area, 3),
            "perimeter_m":          round(perimeter, 3),
            "centroid":             centroid,
            "bounding_box":         bbox,
            "num_vertices":         len(coords),
        },
        "metadata": {
            "tool_name":  "shape_boundary_04",
            "mode":       "mock",
        },
    }


In [5]:
# ── Test all shape types ───────────────────────────────────────────────────────
print("=" * 60)
print("OFFLINE MOCK TESTS — All Shape Types")
print("=" * 60)

ALL_PASS = True
for shape in ["L", "Y", "T", "H", "X", "Z", "O"]:
    result = mock_shape_boundary(shape_type=shape)
    ok = result["success"]
    if ok:
        d = result["data"]
        print(f"  [{shape}]  area={d['area_sqm']:>6.2f}  perim={d['perimeter_m']:>6.2f}"
              f"  verts={d['num_vertices']:>2}  centroid={d['centroid']}  ✓")
    else:
        print(f"  [{shape}]  FAIL — {result['error']}")
        ALL_PASS = False

print()
# ── Test lookup by GUID ────────────────────────────────────────────────────────
guid_result = mock_shape_boundary(shape_guid="00000000-0000-0000-0000-00000000000L")
print("GUID lookup (L-shape GUID):")
print(f"  shape_type={guid_result['data']['shape_type']}  area={guid_result['data']['area_sqm']}  ✓")

print()
# ── Test validation: no input ──────────────────────────────────────────────────
err_result = mock_shape_boundary()
print("Validation — no input provided:")
print(f"  success={err_result['success']}  error='{err_result['error']}'  ✓" if not err_result["success"] else "  UNEXPECTED SUCCESS")

# ── Test validation: unknown shape ─────────────────────────────────────────────
bad_result = mock_shape_boundary(shape_type="Q")
print("Validation — unknown shape 'Q':")
print(f"  success={bad_result['success']}  error='{bad_result['error']}'  ✓" if not bad_result["success"] else "  UNEXPECTED SUCCESS")

print()
print("ALL OFFLINE TESTS PASSED ✓" if ALL_PASS else "SOME TESTS FAILED ✗")


OFFLINE MOCK TESTS — All Shape Types
  [L]  area=  4.00  perim= 10.00  verts= 7  centroid=[0.8571428571428571, 1.1428571428571428]  ✓
  [Y]  area=  5.00  perim= 10.83  verts=10  centroid=[1.0, 1.8]  ✓
  [T]  area=  5.00  perim= 12.00  verts= 9  centroid=[1.3333333333333333, 1.1111111111111112]  ✓
  [H]  area=  7.00  perim= 16.00  verts=13  centroid=[1.3846153846153846, 1.3846153846153846]  ✓
  [X]  area=  5.00  perim= 12.00  verts=13  centroid=[1.4615384615384615, 1.3846153846153846]  ✓
  [Z]  area=  5.00  perim= 17.66  verts= 9  centroid=[1.3333333333333333, 1.3333333333333333]  ✓
  [O]  area= 14.00  perim= 13.66  verts= 9  centroid=[1.7777777777777777, 1.8888888888888888]  ✓

GUID lookup (L-shape GUID):
  shape_type=L  area=4.0  ✓

Validation — no input provided:
  success=False  error='Provide shape_type or shape_guid.'  ✓
Validation — unknown shape 'Q':
  success=False  error='Unknown shape: 'Q''  ✓

ALL OFFLINE TESTS PASSED ✓


## 5. MCP Connection Test

Connect to Swiftlet on `localhost:3001` and list all registered tools.  
**Rhino must be open with the Swiftlet MCP plugin running for this cell to succeed.**


In [6]:
import httpx
from mcp_client import McpClient

mcp: McpClient | None = None
discovered_tools: list[dict] = []

try:
    mcp = McpClient(endpoint=MCP_ENDPOINT, timeout_seconds=10.0)
    mcp.initialize()
    discovered_tools = mcp.list_tools()

    print(f"Connected to MCP at: {MCP_ENDPOINT}")
    print(f"Discovered {len(discovered_tools)} tool(s):\n")
    for t in discovered_tools:
        print(f"  • {t.get('name', 'unnamed')}")
        desc = t.get("description", "")
        if desc:
            print(f"      {desc[:90]}")

    # Check whether our target tool is already registered in Grasshopper
    names = {t.get("name") for t in discovered_tools}
    if "shape_boundary_04" in names:
        print("\nshape_boundary_04 is REGISTERED ✓")
    else:
        print("\nshape_boundary_04 not yet registered — add the Grasshopper component first.")
        print("Existing tools:", sorted(names))

except httpx.ConnectError:
    print(f"Cannot reach MCP server at {MCP_ENDPOINT}")
    print("Make sure Rhino is open and Swiftlet is running.")
except Exception as exc:
    print(f"MCP error: {exc}")


Connected to MCP at: http://localhost:3001/mcp/
Discovered 3 tool(s):

  • site_boundary_generation
      Generates procedural site boundaries using area, proportion, distortion, subdivision, and 
  • Usable_site_area
      area of Usable site area
  • parametric_shape_generator
      Generates building shape boundaries using area, proportion, distortion, subdivision, and r

shape_boundary_04 not yet registered — add the Grasshopper component first.
Existing tools: ['Usable_site_area', 'parametric_shape_generator', 'site_boundary_generation']


In [14]:
import httpx, json, time

TARGET_TOOL = "shape_boundary_04"
ENDPOINT    = MCP_ENDPOINT          # http://localhost:3001/mcp/

# ── Helper: raw JSON-RPC POST ─────────────────────────────────────────────────
def raw_rpc(method: str, params: dict | None = None) -> tuple[int, dict]:
    payload = {"jsonrpc": "2.0", "id": 1, "method": method}
    if params:
        payload["params"] = params
    r = httpx.post(ENDPOINT, json=payload, timeout=10.0)
    return r.status_code, r.json()

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1 — List all registered tools
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print(f"MCP endpoint : {ENDPOINT}")
print("=" * 60)

status, body = raw_rpc("tools/list")
tools = body.get("result", {}).get("tools", [])

print(f"\nHTTP {status} — {len(tools)} tool(s) registered in Swiftlet:\n")
for t in tools:
    registered = "✓" if t["name"] == TARGET_TOOL else "·"
    print(f"  {registered}  {t['name']}")

our_tool = next((t for t in tools if t["name"] == TARGET_TOOL), None)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Call shape_boundary_04 with shape_type = "L"
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 60}")
print(f"CALLING: {TARGET_TOOL}  →  shape_type='L'")
print("=" * 60)

t0 = time.perf_counter()
try:
    status2, body2 = raw_rpc(
        "tools/call",
        {"name": TARGET_TOOL, "arguments": {"shape_type": "L"}}
    )
    elapsed = (time.perf_counter() - t0) * 1000
    print(f"\nHTTP {status2}  ({elapsed:.0f} ms)")
    print(json.dumps(body2, indent=2))

    if "error" in body2:
        err = body2["error"]
        code = err.get("code")
        msg  = err.get("message", "")
        print(f"\n🔴  TOOL NOT READY")
        if code == -32602:
            print(f"   Reason : '{TARGET_TOOL}' is not registered in Grasshopper yet.")
            print("   Action : Add the Swiftlet MCP Tool component in GH and wire it up.")
        else:
            print(f"   Reason : {msg}")
    else:
        content = body2.get("result", {}).get("content", [])
        print(f"\n🟢  TOOL IS WORKING")
        for item in content:
            if item.get("type") == "text":
                try:
                    print(json.dumps(json.loads(item["text"]), indent=2))
                except Exception:
                    print(item["text"])

except httpx.ConnectError:
    print("\n🔴  Cannot reach Swiftlet — is Rhino open with Swiftlet running?")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Call shape_boundary_04 with a GUID
# ═══════════════════════════════════════════════════════════════════════════════
# Replace the GUID below with the actual Rhino object GUID from your model.
TEST_GUID = "00000000-0000-0000-0000-000000000000"  # ← paste real GUID here

print(f"\n{'=' * 60}")
print(f"CALLING: {TARGET_TOOL}  →  shape_guid='{TEST_GUID}'")
print("=" * 60)

try:
    status3, body3 = raw_rpc(
        "tools/call",
        {"name": TARGET_TOOL, "arguments": {"shape_guid": TEST_GUID}}
    )
    print(f"\nHTTP {status3}")
    print(json.dumps(body3, indent=2))
except httpx.ConnectError:
    print("Cannot reach Swiftlet.")


MCP endpoint : http://localhost:3001/mcp/

HTTP 200 — 3 tool(s) registered in Swiftlet:

  ·  site_boundary_generation
  ·  Usable_site_area
  ·  parametric_shape_generator

CALLING: shape_boundary_04  →  shape_type='L'

HTTP 200  (219 ms)
{
  "jsonrpc": "2.0",
  "id": 1,
  "error": {
    "code": -32602,
    "message": "Unknown tool: shape_boundary_04"
  }
}

🔴  TOOL NOT READY
   Reason : 'shape_boundary_04' is not registered in Grasshopper yet.
   Action : Add the Swiftlet MCP Tool component in GH and wire it up.

CALLING: shape_boundary_04  →  shape_guid='00000000-0000-0000-0000-000000000000'

HTTP 200
{
  "jsonrpc": "2.0",
  "id": 1,
  "error": {
    "code": -32602,
    "message": "Unknown tool: shape_boundary_04"
  }
}


## 6. Live MCP Tool Call — `shape_boundary_04`

Call the tool with **shape_type** (text input) and then again with **shape_guid** (GUID input).  
If Rhino is not running, the call will raise a connection error — that is expected.


In [8]:
TOOL_NAME = "shape_boundary_04"

def call_shape_boundary(mcp_client: McpClient | None,
                        shape_type: str | None = None,
                        shape_guid: str | None = None) -> dict:
    """
    Call shape_boundary_04 via MCP.
    Falls back to the local mock if:
      - MCP client is unavailable, or
      - the tool is not yet registered in Grasshopper.
    """
    args: dict = {}
    if shape_guid:
        args["shape_guid"] = shape_guid
    elif shape_type:
        args["shape_type"] = shape_type.upper()
    else:
        return {"success": False, "error": "Provide shape_type or shape_guid."}

    if mcp_client is not None:
        # Check whether the tool is registered before calling
        registered = {t.get("name") for t in discovered_tools}
        if TOOL_NAME in registered:
            try:
                raw = mcp_client.call_tool(TOOL_NAME, args)
                return json.loads(raw) if raw.strip().startswith("{") else {"success": True, "raw": raw}
            except Exception as exc:
                return {"success": False, "error": str(exc)}
        else:
            print(f"[MOCK fallback] '{TOOL_NAME}' not in Rhino yet — using mock.")
    else:
        print("[MOCK fallback] MCP not connected — using mock response.")

    return mock_shape_boundary(shape_type=shape_type, shape_guid=shape_guid)


# ── Test 1: shape_type = "L" ──────────────────────────────────────────────────
print("Test 1 — shape_type = 'L'")
result_L = call_shape_boundary(mcp, shape_type="L")
print(json.dumps(result_L, indent=2))


Test 1 — shape_type = 'L'
[MOCK fallback] 'shape_boundary_04' not in Rhino yet — using mock.
{
  "success": true,
  "data": {
    "shape_type": "L",
    "shape_guid": "00000000-0000-0000-0000-00000000000L",
    "boundary_coordinates": [
      [
        0,
        0
      ],
      [
        2,
        0
      ],
      [
        2,
        1
      ],
      [
        1,
        1
      ],
      [
        1,
        3
      ],
      [
        0,
        3
      ],
      [
        0,
        0
      ]
    ],
    "area_sqm": 4.0,
    "perimeter_m": 10.0,
    "centroid": [
      0.8571428571428571,
      1.1428571428571428
    ],
    "bounding_box": {
      "min": [
        0,
        0
      ],
      "max": [
        2,
        3
      ]
    },
    "num_vertices": 7
  },
  "metadata": {
    "tool_name": "shape_boundary_04",
    "mode": "mock"
  }
}


In [9]:
# ── Test 2: shape_type = "T" ──────────────────────────────────────────────────
print("Test 2 — shape_type = 'T'")
result_T = call_shape_boundary(mcp, shape_type="T")
print(json.dumps(result_T, indent=2))

# ── Test 3: shape_guid (replace with a real GUID from your Rhino model) ───────
# Paste the actual GUID of your Rhino curve here:
MY_SHAPE_GUID = "00000000-0000-0000-0000-00000000000H"   # ← replace with real GUID

print("\nTest 3 — shape_guid (H-shape placeholder GUID)")
result_guid = call_shape_boundary(mcp, shape_guid=MY_SHAPE_GUID)
print(json.dumps(result_guid, indent=2))


Test 2 — shape_type = 'T'
[MOCK fallback] 'shape_boundary_04' not in Rhino yet — using mock.
{
  "success": true,
  "data": {
    "shape_type": "T",
    "shape_guid": "00000000-0000-0000-0000-00000000000T",
    "boundary_coordinates": [
      [
        0,
        0
      ],
      [
        3,
        0
      ],
      [
        3,
        1
      ],
      [
        2,
        1
      ],
      [
        2,
        3
      ],
      [
        1,
        3
      ],
      [
        1,
        1
      ],
      [
        0,
        1
      ],
      [
        0,
        0
      ]
    ],
    "area_sqm": 5.0,
    "perimeter_m": 12.0,
    "centroid": [
      1.3333333333333333,
      1.1111111111111112
    ],
    "bounding_box": {
      "min": [
        0,
        0
      ],
      "max": [
        3,
        3
      ]
    },
    "num_vertices": 9
  },
  "metadata": {
    "tool_name": "shape_boundary_04",
    "mode": "mock"
  }
}

Test 3 — shape_guid (H-shape placeholder GUID)
[MOCK fallback] 'shap

## 7. LLM Integration — Cloudflare + Tool-Call Parsing

Sends a natural-language prompt to the Cloudflare LLM.  
The LLM is given the `shape_boundary_04` schema in its system prompt and must respond with a JSON tool-call.  
We then execute it and return the result.


In [12]:
import re

# ── System prompt — inject tool schema ────────────────────────────────────────
SYSTEM_PROMPT = f"""You are a building design AI assistant.

You have access to ONE tool:

{json.dumps(SHAPE_BOUNDARY_TOOL_SCHEMA, indent=2)}

RULES:
- When the user asks about a building shape boundary, call shape_boundary_04.
- shape_type must be one of: L, Y, T, H, X, Z, O.
- If the user provides a Rhino GUID, use shape_guid instead.
- Respond ONLY with a JSON tool-call object — no prose, no markdown fences.

Example response:
{{
  "action": "tool",
  "tool_calls": [
    {{
      "name": "shape_boundary_04",
      "arguments": {{
        "shape_type": "L"
      }}
    }}
  ]
}}
"""

def call_llm(user_message: str) -> str:
    """POST to Cloudflare Workers AI chat completions endpoint."""
    url = f"{CF_BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {CF_API_TOKEN}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": CF_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        "max_tokens": 300,
        "temperature": 0.0,
    }
    response = httpx.post(url, json=payload, headers=headers, timeout=30.0)
    response.raise_for_status()
    content = response.json()["choices"][0]["message"]["content"]
    # Cloudflare may return content already parsed as a dict; normalise to str
    if isinstance(content, dict):
        return json.dumps(content)
    return str(content)


def extract_tool_call(llm_response: str) -> dict | None:
    """
    Parse the LLM's JSON response and extract the first tool_call.
    Strips markdown fences if the model wraps the JSON.
    """
    # llm_response is always a str at this point
    cleaned = re.sub(r"```(?:json)?|```", "", llm_response).strip()
    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        return None
    calls = parsed.get("tool_calls", [])
    return calls[0] if calls else None


In [13]:
def run_llm_tool_loop(user_prompt: str) -> None:
    """Full round-trip: LLM → parse tool call → execute → print result."""
    print("=" * 60)
    print(f"USER: {user_prompt}")
    print("=" * 60)

    # Step 1: Ask the LLM
    try:
        llm_raw = call_llm(user_prompt)
    except httpx.HTTPStatusError as exc:
        print(f"LLM HTTP error {exc.response.status_code}: {exc.response.text[:300]}")
        return
    except httpx.ConnectError:
        print("Cannot reach Cloudflare API. Check CF credentials and network.")
        return

    print(f"\nLLM raw response:\n{llm_raw}\n")

    # Step 2: Parse tool call
    tool_call = extract_tool_call(llm_raw)
    if not tool_call:
        print("LLM did not return a recognisable tool_call — printing raw response above.")
        return

    tool_name = tool_call.get("name")
    tool_args = tool_call.get("arguments", {})
    print(f"Parsed tool call:  name={tool_name}  args={tool_args}")

    # Step 3: Execute
    if tool_name == TOOL_NAME:
        result = call_shape_boundary(
            mcp,
            shape_type=tool_args.get("shape_type"),
            shape_guid=tool_args.get("shape_guid"),
        )
        print("\nTool result:")
        print(json.dumps(result, indent=2))
    else:
        print(f"Unknown tool '{tool_name}' — not executing.")


# ── LLM Test prompts ──────────────────────────────────────────────────────────
TEST_PROMPTS = [
    "What is the boundary of an L-shaped building?",
    "Show me the footprint of a T-shaped building.",
    "Get the shape boundary for an X-type building form.",
]

for prompt in TEST_PROMPTS:
    run_llm_tool_loop(prompt)
    print()


USER: What is the boundary of an L-shaped building?

LLM raw response:
{"action": "tool", "tool_calls": [{"name": "shape_boundary_04", "arguments": {"shape_type": "L"}}]}

Parsed tool call:  name=shape_boundary_04  args={'shape_type': 'L'}
[MOCK fallback] 'shape_boundary_04' not in Rhino yet — using mock.

Tool result:
{
  "success": true,
  "data": {
    "shape_type": "L",
    "shape_guid": "00000000-0000-0000-0000-00000000000L",
    "boundary_coordinates": [
      [
        0,
        0
      ],
      [
        2,
        0
      ],
      [
        2,
        1
      ],
      [
        1,
        1
      ],
      [
        1,
        3
      ],
      [
        0,
        3
      ],
      [
        0,
        0
      ]
    ],
    "area_sqm": 4.0,
    "perimeter_m": 10.0,
    "centroid": [
      0.8571428571428571,
      1.1428571428571428
    ],
    "bounding_box": {
      "min": [
        0,
        0
      ],
      "max": [
        2,
        3
      ]
    },
    "num_vertices": 7
 

KeyboardInterrupt: 

## 8. Grasshopper Component — Implementation Reference

When you add the `shape_boundary_04` component in Grasshopper, the Python Script component should implement the logic below.

### Input panel (JSON, stored on a Panel node)
```json
{
  "L": "<guid-of-L-curve>",
  "Y": "<guid-of-Y-curve>",
  "T": "<guid-of-T-curve>",
  "H": "<guid-of-H-curve>",
  "X": "<guid-of-X-curve>",
  "Z": "<guid-of-Z-curve>",
  "O": "<guid-of-O-curve>"
}
```

### GHPython component inputs
| Name | Type | Description |
|---|---|---|
| `input_json` | Text | Raw JSON from Swiftlet MCP call |
| `registry_json` | Text | Shape→GUID mapping (Panel above) |

### GHPython script
```python
import json, System
import Rhino.Geometry as rg
import Rhino

data     = json.loads(input_json)
registry = json.loads(registry_json)

shape_guid_str = data.get("shape_guid")
shape_type     = data.get("shape_type", "").upper()

# Resolve GUID
if not shape_guid_str:
    shape_guid_str = registry.get(shape_type)

if not shape_guid_str:
    output_json = json.dumps({"success": False, "error": f"No GUID for shape '{shape_type}'"})
else:
    guid = System.Guid(shape_guid_str)
    obj  = Rhino.RhinoDoc.ActiveDoc.Objects.FindId(guid)
    crv  = obj.Geometry if obj else None

    if not crv:
        output_json = json.dumps({"success": False, "error": "GUID not found in document"})
    else:
        area_props = rg.AreaMassProperties.Compute(crv)
        mp = rg.Polyline()
        crv.TryGetPolyline(mp)
        pts = [[p.X, p.Y] for p in mp]

        output_json = json.dumps({
            "success": True,
            "data": {
                "shape_type":           shape_type or "custom",
                "shape_guid":           shape_guid_str,
                "boundary_coordinates": pts,
                "area_sqm":             round(area_props.Area, 3),
                "centroid":             [round(area_props.Centroid.X, 3),
                                         round(area_props.Centroid.Y, 3)],
            },
            "metadata": {"tool_name": "shape_boundary_04", "mode": "live"}
        })

# Output wire → Swiftlet result
a = output_json
```

> **Steps to register in Swiftlet:**  
> 1. Add a Swiftlet `MCP Tool` component, set name = `shape_boundary_04`  
> 2. Paste the tool schema JSON (from Cell 3) into its description slot  
> 3. Wire the GHPython component to the Tool component's input/output  
> 4. Restart Swiftlet — the tool will appear in `list_tools`
